In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import pickle
import copy
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from pathlib import Path

from imports import *
from config import dir_config, ephys_config
from src.utils import dpca_utils, dpca_plot_utils
import src.utils.ephys_utils as ephys_utils

## Load data

In [ ]:
compiled_dir  = Path(dir_config.data.compiled)
processed_dir = Path(dir_config.data.processed)

session_to_exclude = ["210210_GP_JP", "241209_GP_TZ"]

session_metadata = pd.read_csv(Path(processed_dir, "sessions_metadata.csv"))
session_metadata = session_metadata[~session_metadata.session_id.isin(session_to_exclude)].reset_index(drop=True)

neuron_metadata = pd.read_csv(Path(processed_dir, "neuron_metadata.csv"))
neuron_metadata = neuron_metadata[~neuron_metadata.session_id.isin(session_to_exclude)].reset_index(drop=True)

with open(Path(processed_dir, "glm_hmm_models", "glm_hmm_masked_final.pkl"), "rb") as f:
    glm_hmm = pickle.load(f)
glm_hmm_original = copy.deepcopy(glm_hmm)

with open(Path(processed_dir, "ephys_neuron_wise.pkl"), "rb") as f:
    ephys = pickle.load(f)

## Extract trial info

In [ ]:
data = glm_hmm["data"]

biased_state_trial_info, unbiased_state_trial_info, state_occupancy = \
    dpca_utils.extract_hmm_state_trial_info(session_metadata, glm_hmm_original, data,
                                            compiled_dir=compiled_dir)

# run after extract_hmm_state_trial_info so awayRF sign flip is already applied
equal_block_trial_info, unequal_block_trial_info = \
    dpca_utils.extract_block_trial_info(data, session_metadata["session_id"])

## Setup

In [ ]:
sessions   = session_metadata.session_id[session_metadata.prior_direction == "toRF"]
alignments = list(ephys_config["alignment_settings_GP"].keys())

neuron_ids = dpca_utils.get_neuron_ids(neuron_metadata, sessions)

# concatenate equal + unequal block trials per session
all_block_trial_info = {
    sid: pd.concat([equal_block_trial_info[sid], unequal_block_trial_info[sid]],
                   ignore_index=True)
    for sid in sessions
}
all_trial_info = {"all": all_block_trial_info}

COH_LIST   = [-0.2, -0.06, 0.0, 0.06, 0.2]
COH_COLORS = ["#1a3a6b", "#9ec8e8", "#888888", "#f4a460", "#8b1a0e"]
COH_LABELS = ["-20%", "-6%", "0%", "+6%", "+20%"]

condition_dict_eq = {
    "state_values": ["all"],
    "coherences":   COH_LIST,
    "choices":      ["awayRF", "toRF"],
}

## Build data matrix

`get_trial_num` uses `np.abs(stimulus) == coherence` which ignores sign.
Monkey-patch to use exact signed match, then restore.

In [ ]:
_orig_get_trial_num = ephys_utils.get_trial_num

def _get_trial_num_signed(trial_data, coherence, choice, outcome=None):
    trial_data = trial_data[~np.isnan(trial_data.reaction_time)]
    idx = (trial_data.stimulus == coherence) & (trial_data.choices == choice)
    return np.array(trial_data["trial_num"][idx].values.reshape(-1, 1))

ephys_utils.get_trial_num = _get_trial_num_signed

avg_data, tw_data = dpca_utils.create_dpca_matrix(
    sessions, condition_dict_eq, neuron_ids,
    all_trial_info, neuron_metadata, ephys, ephys_config,
    condition_type="blocks", outcome=None,
)

ephys_utils.get_trial_num = _orig_get_trial_num
print("avg_data shapes:", {a: avg_data[a].shape for a in alignments})

## Exclusion and trial count check

Exclude neurons with < 2 trials in any ±20% condition.  
COH_LIST indices: 0=-20%, 1=-6%, 2=0%, 3=+6%, 4=+20%

In [ ]:
# trial counts per (neuron, coh, choice) — baseline has full time coverage
valid  = ~np.all(np.isnan(tw_data["baseline"]), axis=-1)   # (n_trials, n_neurons, 1, 5, 2)
counts = valid.sum(axis=0)[:, 0, :, :]                      # (n_neurons, 5, 2)

# ±20% conditions: indices 0 (-20%) and 4 (+20%), both choices → 4 conditions
counts_20 = np.stack([counts[:, 0, 0],   # -20% awayRF
                      counts[:, 0, 1],   # -20% toRF
                      counts[:, 4, 0],   # +20% awayRF
                      counts[:, 4, 1]],  # +20% toRF
                     axis=1)             # (n_neurons, 4)
cond_labels_20 = ["-20%/awayRF", "-20%/toRF", "+20%/awayRF", "+20%/toRF"]

min_per_neuron = counts_20.min(axis=1)
bad_mask = min_per_neuron < 4
neuron_ids_eq = neuron_ids[~bad_mask]
avg_filt = {a: v[~bad_mask] for a, v in avg_data.items()}
tw_filt  = {a: v[:, ~bad_mask] for a, v in tw_data.items()}

print(f"\nNeurons excluded (min < 4 in any ±20% cond): {bad_mask.sum()}")
print(f"Neurons remaining                           : {(~bad_mask).sum()}")

## Clean data and squeeze state dimension

In [ ]:
fit_avg, fit_tw, full_avg, full_tw = dpca_utils.clean_dpca_data(avg_filt, tw_filt, alignments)

# n_states=1 — squeeze before fitting: (n_neurons, 1, 5, 2, T) → (n_neurons, 5, 2, T)
fit_avg_sq  = {a: v[:, 0] for a, v in fit_avg.items()}
fit_tw_sq   = {a: v[:, :, 0] for a, v in fit_tw.items()}
full_avg_sq = {a: v[:, 0] for a, v in full_avg.items()}

## Fit dPCA

Data is `(n_neurons, 5_coh, 2_choices, n_time)` → labels `'sct'`.

In [ ]:
dpca_results_sct = dpca_utils.fit_dpca_all_alignments(
    fit_avg_sq, fit_tw_sq, alignments,
    n_components=3,
    marginalization_keys=["s", "c", "t"],
)
time_axes_sct     = dpca_utils.build_time_axes(fit_avg_sq, ephys_config)
full_time_axes_sct = dpca_utils.build_time_axes(full_avg_sq, ephys_config)

## Variance explained

In [ ]:
fig = dpca_plot_utils.plot_variance_explained(dpca_results_sct, alignments, ["s", "c", "t"], n_components=3)
plt.show()

## Self-projection

In [ ]:
significance_masks_sct = {}
for alignment in alignments:
    dpca_model = dpca_results_sct[alignment]["model"]
    print(f"Computing significance masks for {alignment}...")
    significance_masks_sct[alignment], _, _ = dpca_utils.dpca_significance_analysis(
        copy.deepcopy(dpca_model), fit_avg_sq[alignment], fit_tw_sq[alignment],
        n_shuffles=100, n_splits=50, n_consecutive=1,
        keys=["s", "c"],
        key_groups={"s": [[0],[1,2,3], [4]]},
        smooth_sigma=10,
    )

In [ ]:
n_coh = len(COH_LIST)
legend_handles = (
    [Line2D([], [], color=COH_COLORS[i], lw=1.5, label=COH_LABELS[i]) for i in range(n_coh)] +
    [Line2D([], [], color="k", lw=1.5, ls="-",  label="toRF"),
     Line2D([], [], color="k", lw=1.0, ls="--", label="awayRF")]
)

margs_to_plot = ["s", "c"]
PC = 0
n_rows, n_cols = len(margs_to_plot), len(alignments)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))

for col, alignment in enumerate(alignments):
    Z    = dpca_results_sct[alignment]["transformed_data"]
    time = time_axes_sct[alignment]
    for row, marg in enumerate(margs_to_plot):
        ax = axes[row, col]
        data_pc = Z[marg][PC]
        for ci in range(n_coh):
            ax.plot(time, data_pc[ci, 1, :], color=COH_COLORS[ci], lw=1.5, ls="-")
            ax.plot(time, data_pc[ci, 0, :], color=COH_COLORS[ci], lw=1.0, ls="--")
        if marg in significance_masks_sct.get(alignment, {}):
            dpca_plot_utils.bar_significance(ax, time, significance_masks_sct[alignment][marg][PC])
        ax.axvline(0, color="k", ls="--", lw=0.8)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        if row == 0:
            ax.set_title(dpca_plot_utils.ALIGN_LABELS.get(alignment, alignment), fontsize=13)
        if col == 0:
            ax.set_ylabel(f"{dpca_plot_utils.MARG_LABELS.get(marg, marg)} PC{PC + 1}", fontsize=12)

fig.legend(handles=legend_handles, bbox_to_anchor=(1.01, 0.9), loc="upper left", fontsize=10)
fig.suptitle("Self-projection — all blocks, signed coherence", y=1.01)
plt.tight_layout()
plt.show()
fig.savefig(Path("../dissemination/dpca/dpca_toRF_session_signed_coh") / "self_projection.png", dpi=150, bbox_inches="tight")

## Correct vs error trial projections

Project correct and error trials separately onto the already-fitted dPCA axes.
Many (coh, choice) cells will be NaN for error trials — handled by `clean_dpca_data`.

In [ ]:
def _get_trial_num_signed_outcome(trial_data, coherence, choice, outcome=None):
    trial_data = trial_data[~np.isnan(trial_data.reaction_time)]
    idx = (trial_data.stimulus == coherence) & (trial_data.choices == choice)
    if outcome is not None and coherence != 0:
        sign = trial_data.stimulus * (trial_data.choices * 2 - 1)
        if outcome == 1:
            idx = idx & (sign > 0)
        elif outcome == 0:
            idx = idx & (sign < 0)
    return np.array(trial_data["trial_num"][idx].values.reshape(-1, 1))

ephys_utils.get_trial_num = _get_trial_num_signed_outcome

avg_correct, tw_correct = dpca_utils.create_dpca_matrix(
    sessions, condition_dict_eq, neuron_ids_eq,
    all_trial_info, neuron_metadata, ephys, ephys_config,
    condition_type="blocks", outcome=1,
)
avg_error, tw_error = dpca_utils.create_dpca_matrix(
    sessions, condition_dict_eq, neuron_ids_eq,
    all_trial_info, neuron_metadata, ephys, ephys_config,
    condition_type="blocks", outcome=0,
)

ephys_utils.get_trial_num = _orig_get_trial_num
print("correct:", {a: avg_correct[a].shape for a in alignments})
print("error:  ", {a: avg_error[a].shape for a in alignments})

In [ ]:
# Clean (handles NaN-only conditions for sparse error trials)
fit_avg_c, fit_tw_c, _, _ = dpca_utils.clean_dpca_data(avg_correct, tw_correct, alignments, compute_full=False)
fit_avg_e, fit_tw_e, _, _ = dpca_utils.clean_dpca_data(avg_error,   tw_error,   alignments, compute_full=False)

# Squeeze state dim
fit_avg_c_sq = {a: v[:, 0]    for a, v in fit_avg_c.items()}
fit_avg_e_sq = {a: v[:, 0]    for a, v in fit_avg_e.items()}
fit_tw_c_sq  = {a: v[:, :, 0] for a, v in fit_tw_c.items()}
fit_tw_e_sq  = {a: v[:, :, 0] for a, v in fit_tw_e.items()}

# Time axes matching cleaned data
time_axes_correct = dpca_utils.build_time_axes(fit_avg_c_sq, ephys_config)
time_axes_error   = dpca_utils.build_time_axes(fit_avg_e_sq, ephys_config)

# Project onto fitted dPCA axes (no refitting)
Z_correct, Z_error = {}, {}
for alignment in alignments:
    model = dpca_results_sct[alignment]["model"]
    _, Z_correct[alignment] = dpca_utils.dpca_transform(model, fit_avg_c_sq[alignment])
    _, Z_error[alignment]   = dpca_utils.dpca_transform(model, fit_avg_e_sq[alignment])

In [ ]:
for trial_label, fit_avg_sq_sub, fit_tw_sq_sub, masks_dict in [
    ("correct", fit_avg_c_sq, fit_tw_c_sq, {}),
    ("error",   fit_avg_e_sq, fit_tw_e_sq, {}),
]:
    for alignment in alignments:
        dpca_model = dpca_results_sct[alignment]["model"]
        print(f"Computing significance masks — {trial_label} / {alignment}...")
        masks_dict[alignment], _, _ = dpca_utils.dpca_significance_analysis(
            copy.deepcopy(dpca_model),
            fit_avg_sq_sub[alignment], fit_tw_sq_sub[alignment],
            n_shuffles=100, n_splits=50, n_consecutive=1,
            keys=["s", "c"],
            key_groups={"s": [[0],[1,2,3], [4]]},
            smooth_sigma=10,
            refit=False,
        )
    if trial_label == "correct":
        significance_masks_correct = masks_dict
    else:
        significance_masks_error = masks_dict

In [ ]:
PC = 0

for trial_label, Z_data, time_ax, sig_masks in [
    ("correct", Z_correct, time_axes_correct, significance_masks_correct),
    ("error",   Z_error,   time_axes_error,   significance_masks_error),
]:
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
    for col, alignment in enumerate(alignments):
        Z    = Z_data[alignment]
        time = time_ax[alignment]
        for row, marg in enumerate(margs_to_plot):
            ax = axes[row, col]
            data_pc = Z[marg][PC]
            for ci in range(n_coh):
                ax.plot(time, data_pc[ci, 1, :], color=COH_COLORS[ci], lw=1.5, ls="-")
                ax.plot(time, data_pc[ci, 0, :], color=COH_COLORS[ci], lw=1.0, ls="--")
            if marg in sig_masks.get(alignment, {}):
                dpca_plot_utils.bar_significance(ax, time, sig_masks[alignment][marg][PC])
            ax.axvline(0, color="k", ls="--", lw=0.8)
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)
            if row == 0:
                ax.set_title(dpca_plot_utils.ALIGN_LABELS.get(alignment, alignment), fontsize=13)
            if col == 0:
                ax.set_ylabel(f"{dpca_plot_utils.MARG_LABELS.get(marg, marg)} PC{PC + 1}", fontsize=12)
    fig.legend(handles=legend_handles, bbox_to_anchor=(1.01, 0.9), loc="upper left", fontsize=10)
    fig.suptitle(f"Projection — {trial_label} trials, signed coherence", y=1.01)
    plt.tight_layout()
    plt.show()
    fig.savefig(Path("../dissemination/dpca/dpca_toRF_session_signed_coh") / f"self_projection_{trial_label}.png", dpi=150, bbox_inches="tight")